<a href="https://colab.research.google.com/github/Nikhilsankhyan0/Agrisheild-AI/blob/main/regression_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

Mounted at /content/drive


In [2]:
x_train = pd.read_csv(
    "/content/drive/MyDrive/X_train_encoded.csv"
)

x_val = pd.read_csv(
    "/content/drive/MyDrive/X_val_encoded.csv"
)

x_test = pd.read_csv(
    "/content/drive/MyDrive/X_test_encoded.csv"
)

In [3]:
y_train_reg = pd.read_csv(
    "/content/drive/MyDrive/y_train_reg.csv"
).squeeze()

y_val_reg = pd.read_csv(
    "/content/drive/MyDrive/y_val_reg.csv"
).squeeze()

y_test_reg = pd.read_csv(
    "/content/drive/MyDrive/y_test_reg.csv"
).squeeze()

target to numeric

In [4]:
y_train_reg = pd.to_numeric(y_train_reg)
y_val_reg = pd.to_numeric(y_val_reg)
y_test_reg = pd.to_numeric(y_test_reg)

scale the features

In [5]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_val_scaled = scaler.transform(x_val)

x_test_scaled = scaler.transform(x_test)

train the regression model

In [6]:
baseline_regressor = LinearRegression()

baseline_regressor.fit(
    x_train_scaled,
    y_train_reg
)

LinearRegression()

In [7]:
y_pred_reg = baseline_regressor.predict(
    x_val_scaled
)

In [8]:
mae = mean_absolute_error(
    y_val_reg,
    y_pred_reg
)

rmse = np.sqrt(
    mean_squared_error(
        y_val_reg,
        y_pred_reg
    )
)

r2 = r2_score(
    y_val_reg,
    y_pred_reg
)

In [9]:
print("===== LINEAR REGRESSION BASELINE =====")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

===== LINEAR REGRESSION BASELINE =====
MAE  : 4.1796
RMSE : 5.3001
R²   : 0.8239


In [12]:
feature_importance_reg = pd.DataFrame({
    'data': x_train.columns,
    'importance': baseline_regressor.coef_
})

In [13]:
print(len(feature_importance_reg))

259


In [14]:
top_features_reg = feature_importance_reg.head(20)

In [15]:
print(
    feature_importance_reg
    .sort_values('importance', ascending=False)
    .head(15)
)

                         data  importance
239       Crop_Insurance_True    2.851448
19               Credit_Score    2.663305
241   Government_Subsidy_True    1.948624
22    Repayment_History_Score    1.353713
29    Farm_Productivity_Index    0.886215
3    Farming_Experience_Years    0.865868
240    Crop_Insurance_Unknown    0.709305
31     Financial_Health_Index    0.683395
67          State_West Bengal    0.535280
49                State_Bihar    0.493335
55            State_Karnataka    0.491755
56               State_Kerala    0.482979
57       State_Madhya Pradesh    0.463414
53     State_Himachal Pradesh    0.456892
54            State_Jharkhand    0.418931


In [16]:
print(
    feature_importance_reg
    .sort_values('importance', ascending=True)
    .head(15)
)

                         data  importance
247     Previous_Default_True   -4.271362
20        Existing_Loan_Count   -2.693842
30         Climate_Risk_Index   -1.640253
34        Overall_Risk_Factor   -1.401533
23       Debt_to_Income_Ratio   -0.956771
248  Previous_Default_Unknown   -0.328785
154            District_Solan   -0.301088
9          Weather_Risk_Index   -0.279016
128            District_Mandi   -0.249386
44     Marital_Status_Married   -0.235324
152         District_Siliguri   -0.235165
85           District_Cuttack   -0.231760
45      Marital_Status_Single   -0.230874
105         District_Jabalpur   -0.224568
123            District_Kullu   -0.224135


In [17]:
feature_importance_reg['Absolute_Importance'] = (
    feature_importance_reg['importance'].abs()
)

weak_features_reg = (
    feature_importance_reg
    .sort_values('Absolute_Importance', ascending=True)
    .head(20)
)

print(weak_features_reg)

                             data  importance  Absolute_Importance
119               District_Kollam   -0.000817             0.000817
136              District_Nellore    0.001999             0.001999
115              District_Khammam    0.002077             0.002077
114               District_Karnal    0.002467             0.002467
122            District_Kozhikode    0.002642             0.002642
36                   Gender_Other   -0.002879             0.002879
97              District_Haldwani   -0.003293             0.003293
186           Primary_Crop_Chilli   -0.003477             0.003477
224     Secondary_Crop_Vegetables    0.003678             0.003678
255  Loan_Purpose_Processing Unit   -0.003720             0.003720
188           Primary_Crop_Coffee   -0.004506             0.004506
160              District_Udaipur   -0.004783             0.004783
32       Loan_Affordability_Index    0.005643             0.005643
39     Education_Level_Illiterate   -0.006339             0.00

In [18]:
weak_feature_names = weak_features_reg['data'].tolist()

print("Features to remove:")
print(weak_feature_names)

Features to remove:
['District_Kollam', 'District_Nellore', 'District_Khammam', 'District_Karnal', 'District_Kozhikode', 'Gender_Other', 'District_Haldwani', 'Primary_Crop_Chilli', 'Secondary_Crop_Vegetables', 'Loan_Purpose_Processing Unit', 'Primary_Crop_Coffee', 'District_Udaipur', 'Loan_Affordability_Index', 'Education_Level_Illiterate', 'Climate_Zone_Temperate/Hill', 'District_Hyderabad', 'District_Kanpur', 'District_Rajkot', 'Education_Level_Unknown', 'Secondary_Crop_Gram']


In [19]:
x_train_reduced = x_train.drop(
    columns=weak_feature_names
)

x_val_reduced = x_val.drop(
    columns=weak_feature_names
)

x_test_reduced = x_test.drop(
    columns=weak_feature_names
)

print("Original:", x_train.shape)
print("Reduced :", x_train_reduced.shape)

Original: (14000, 259)
Reduced : (14000, 239)
